In [2]:
from kan import *
import skan
from skan import SKANNetwork
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# device = torch.device('cpu')

cuda


In [3]:
# Train on CIFAR10
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Load CIFAR
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
trainset = torchvision.datasets.CIFAR10(
    root="./cifar-data", train=True, download=True, transform=transform
)
valset = torchvision.datasets.CIFAR10(
    root="./cifar-data", train=False, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170M/170M [00:43<00:00, 3.88MB/s]


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# Define the MLP model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Define the CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten before passing to fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Training function
def train_model(model, trainloader, valloader, epochs=10, lr=0.001, model_type="mlp"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        if model_type == "kan":
            model.speed()
            
        total_loss = 0
        correct, total = 0, 0
        
        progress_bar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            if model_type in ["mlp", "kan", "skan"]:
                images = images.view(images.size(0), -1)  # Reshape for MLP or KAN
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            progress_bar.set_postfix(loss=loss.item())
        
        val_acc = evaluate_model(model, valloader, device, model_type)
        print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {total_loss/len(trainloader):.4f}, Train Acc: {correct/total:.4f}, Val Acc: {val_acc:.4f}")

    return val_acc
    
# Evaluation function
def evaluate_model(model, dataloader, device, model_type="mlp"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            if model_type in ["mlp", "kan", "skan"]:
                images = images.view(images.size(0), -1)  # Reshape for MLP

            if model_type == "kan":
                model.speed()
                
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# Function to count trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [17]:
%%time

# KAN
kan_model = KAN(width=[3 * 32 * 32, 64, 10], device=device)

train_model(kan_model, trainloader, valloader, model_type="kan")

checkpoint directory created: ./model
saving model version 0.0


Epoch 1/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:10<00:00, 73.34it/s, loss=1.84]


Epoch 1/10, Avg Loss: 1.6039, Train Acc: 0.4303, Val Acc: 0.4784


Epoch 2/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:10<00:00, 73.57it/s, loss=1.26]


Epoch 2/10, Avg Loss: 1.3901, Train Acc: 0.5071, Val Acc: 0.5000


...

Epoch 9/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:10<00:00, 74.36it/s, loss=0.856]


Epoch 9/10, Avg Loss: 1.0065, Train Acc: 0.6447, Val Acc: 0.5265


Epoch 10/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:10<00:00, 74.19it/s, loss=0.764]


Epoch 10/10, Avg Loss: 0.9686, Train Acc: 0.6583, Val Acc: 0.5209
CPU times: user 2min 4s, sys: 1.11 s, total: 2min 5s
Wall time: 2min 2s


0.5209

In [21]:
%%time

# SKAN with arctan

def larctan(x, k):
    return k * torch.atan(x)
    
skan_model = SKANNetwork([3 * 32 * 32, 256, 256, 256, 10], basis_function=larctan).to(device)
train_model(skan_model, trainloader, valloader, model_type="skan")

Epoch 1/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:08<00:00, 89.80it/s, loss=1.39]


Epoch 1/10, Avg Loss: 1.7648, Train Acc: 0.3795, Val Acc: 0.4294


Epoch 2/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:08<00:00, 89.66it/s, loss=1.8]


Epoch 2/10, Avg Loss: 1.5989, Train Acc: 0.4409, Val Acc: 0.4494


Epoch 3/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:08<00:00, 88.50it/s, loss=1.63]


...

Epoch 9/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:08<00:00, 89.75it/s, loss=0.821]


Epoch 9/10, Avg Loss: 1.1301, Train Acc: 0.6003, Val Acc: 0.4916


Epoch 10/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:08<00:00, 89.80it/s, loss=1.37]


Epoch 10/10, Avg Loss: 1.0803, Train Acc: 0.6190, Val Acc: 0.4961
CPU times: user 1min 40s, sys: 974 ms, total: 1min 41s
Wall time: 1min 39s


0.4961

In [13]:
%%time

# MLP
mlp_model = MLP()
print("Training MLP Model:")
train_model(mlp_model, trainloader, valloader, model_type="mlp")

Training MLP Model:


Epoch 1/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 120.26it/s, loss=1.58]


Epoch 1/10, Avg Loss: 1.6309, Train Acc: 0.4185, Val Acc: 0.4622


Epoch 2/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 119.80it/s, loss=1.31]


Epoch 2/10, Avg Loss: 1.4285, Train Acc: 0.4956, Val Acc: 0.5054


...

Epoch 9/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 119.08it/s, loss=0.883]


Epoch 9/10, Avg Loss: 0.8516, Train Acc: 0.6977, Val Acc: 0.5362


Epoch 10/10: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 119.78it/s, loss=1.25]


Epoch 10/10, Avg Loss: 0.7798, Train Acc: 0.7199, Val Acc: 0.5309
CPU times: user 1min 16s, sys: 808 ms, total: 1min 17s
Wall time: 1min 15s


0.5309

In [11]:
%%time

# CNN
cnn_model = CNN()
print("\nTraining CNN Model:")
train_model(cnn_model, trainloader, valloader, model_type="cnn")


Training CNN Model:


Epoch 1/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 112.49it/s, loss=1.13]


Epoch 1/10, Avg Loss: 1.3006, Train Acc: 0.5349, Val Acc: 0.6180


Epoch 2/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 115.18it/s, loss=1.2]


Epoch 2/10, Avg Loss: 0.9293, Train Acc: 0.6699, Val Acc: 0.6758


...

Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 114.37it/s, loss=0.0393]


Epoch 9/10, Avg Loss: 0.1500, Train Acc: 0.9487, Val Acc: 0.7186


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:06<00:00, 113.17it/s, loss=0.0713]


Epoch 10/10, Avg Loss: 0.1057, Train Acc: 0.9643, Val Acc: 0.7158


0.7158

In [22]:
print(f"Trainable parameters in KAN : {count_parameters(kan_model):,}")
print(f"Trainable parameters in SKAN: {count_parameters(skan_model):,}")
print(f"Trainable parameters in MLP : {count_parameters(mlp_model):,}")
print(f"Trainable parameters in CNN : {count_parameters(cnn_model):,}")

Trainable parameters in KAN : 2,366,976
Trainable parameters in SKAN: 920,842
Trainable parameters in MLP : 1,707,274
Trainable parameters in CNN : 1,070,794


## KAN Experiments

In [15]:
val_accs = {}

for widths in [[3072, 256, 10], [3072, 512, 10], [3072, 256, 256, 10], [3072, 512, 256, 10]]:
    print("Training for width: ", widths)
    kan_model = KAN(width=widths, device=device)
    val_acc = train_model(kan_model, trainloader, valloader, is_kan=True)
    val_accs[str(widths)] = val_acc

Training for width:  [3072, 256, 10]
checkpoint directory created: ./model
saving model version 0.0


Epoch 1/10: 100%|██████████████████████████████████████████████████████████| 782/782 [00:23<00:00, 33.05it/s, loss=1.32]


Epoch 1/10, Avg Loss: 1.5633, Train Acc: 0.4436, Val Acc: 0.5000


Epoch 2/10: 100%|██████████████████████████████████████████████████████████| 782/782 [00:23<00:00, 33.07it/s, loss=1.22]


Epoch 2/10, Avg Loss: 1.3116, Train Acc: 0.5353, Val Acc: 0.5353


...

Epoch 8/10, Avg Loss: 0.4886, Train Acc: 0.8261, Val Acc: 0.5518


Epoch 9/10: 100%|█████████████████████████████████████████████████████████| 782/782 [00:32<00:00, 24.35it/s, loss=0.566]


Epoch 9/10, Avg Loss: 0.4020, Train Acc: 0.8559, Val Acc: 0.5403


Epoch 10/10: 100%|████████████████████████████████████████████████████████| 782/782 [00:36<00:00, 21.71it/s, loss=0.278]


Epoch 10/10, Avg Loss: 0.3478, Train Acc: 0.8767, Val Acc: 0.5488


In [16]:
val_accs

{'[[3072, 0], [256, 0], [10, 0]]': 0.5468,
 '[[3072, 0], [512, 0], [10, 0]]': 0.5442,
 '[[3072, 0], [256, 0], [256, 0], [10, 0]]': 0.545,
 '[[3072, 0], [512, 0], [256, 0], [10, 0]]': 0.5488}

In [18]:
val_accs = {}

for widths in [[3072, 128, 128, 128, 128, 128, 10], [3072, 256, 256, 256, 10]]:
    print("Training for width: ", widths)
    kan_model = KAN(width=widths, device=device)
    val_acc = train_model(kan_model, trainloader, valloader, is_kan=True)
    val_accs[str(widths)] = val_acc

Training for width:  [3072, 128, 128, 128, 128, 128, 10]
checkpoint directory created: ./model
saving model version 0.0


Epoch 1/10: 100%|██████████████████████████████████████████████████████████| 782/782 [00:38<00:00, 20.54it/s, loss=1.59]


Epoch 1/10, Avg Loss: 1.6675, Train Acc: 0.3975, Val Acc: 0.4625


Epoch 2/10: 100%|██████████████████████████████████████████████████████████| 782/782 [00:35<00:00, 21.82it/s, loss=1.31]


Epoch 2/10, Avg Loss: 1.3991, Train Acc: 0.4987, Val Acc: 0.5065


...

Epoch 8/10, Avg Loss: 0.6045, Train Acc: 0.7824, Val Acc: 0.5480


Epoch 9/10: 100%|█████████████████████████████████████████████████████████| 782/782 [00:30<00:00, 25.37it/s, loss=0.744]


Epoch 9/10, Avg Loss: 0.5248, Train Acc: 0.8095, Val Acc: 0.5455


Epoch 10/10: 100%|████████████████████████████████████████████████████████| 782/782 [00:30<00:00, 25.42it/s, loss=0.473]


Epoch 10/10, Avg Loss: 0.4496, Train Acc: 0.8363, Val Acc: 0.5479
